In [22]:
import numpy as np
with open('/content/hostel_bois.txt','r',encoding='utf-8') as f:
  content=f.read()
  lines=content.split('\n')


messages=[]
members=[]
days=[]
print('WHATSAPP CHAT REPORT :"Hostel bois 4ever"')
for line in lines:
  line=line.strip()
  if " - " in line and ": " in line:
    date,remaining = line.split(", ", 1)
    time,remaining = remaining.split(" - ", 1)
    name, message = remaining.split(": ", 1)

    data={
        "Date":date,
        "Time":time,
        "Name":name,
        "Messages":message
    }
    messages.append(data)

    if name not in members:
      members.append(name)

    if date not in days:
      days.append(date)
count=0

for msg in messages:
  count=count+1
print("Members:",len(members))
print("Number of messages:",count)
print("Days:",len(days))
print("*" *100)

print("MESSAGES PER PERSON")
message_count={}
for member in members:
  count=0
  for msg in messages:
    if msg["Name"]==member:
      count=count+1
  message_count[member] = count
for member in message_count:
  print(member,":",message_count[member])

count_day={}
for msg in messages:
  date=msg["Date"]
  if date not in count_day:
    count_day[date]=1
  else:
    count_day[date]=count_day[date]+1

busiest_day=""
highest_msg=0
for date in count_day:
  if count_day[date]>highest_msg:
    highest_msg=count_day[date]
    busiest_day=date
print("Busiest day:",busiest_day ,"(Highest messages:",highest_msg,")")
print("*" *100)

heatmap = np.zeros((len(members), 24))
for msg in messages:
    name = msg["Name"]
    hour = int(msg["Time"][:2])

    person = members.index(name)

    heatmap[person][hour] += 1


print("Activity Heatmap")
print("Hour:       00  01  02  03  04  05  06  07  08  09  10  11  12  13  14  15  16  17  18  19  20  21  22  23\n")

for i in range(len(members)):

    print(f"{members[i]:10}", end=": ")

    highest = max(heatmap[i])

    for hour in range(24):

        value = heatmap[i][hour]

        if value == 0:
            print(" . ", end=" ")
        elif value <= highest * 0.25:
            print(" + ", end=" ")
        elif value <= highest * 0.50:
            print(" * ", end=" ")
        elif value <= highest * 0.75:
            print(" # ", end=" ")
        else:
            print(" % ", end=" ")

    print()

print("Note: here denser symbols indicate active days")
print("*" *100)
print("THIS GROUP'S FAVOURITE WORDS")
remove=["i", "is", "the", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "was", "were", "are", "am", "be",
    "been", "being", "this", "that", "these", "those",
    "you", "your", "yours", "we", "our", "ours", "they",
    "their", "them", "he", "his", "him", "she", "her",
    "it", "its", "my", "me", "mine", "so", "about", "at",
    "by", "with", "from", "as", "but", "if", "then",
    "than", "have", "has", "had", "just", "which", "who",
    "what", "when", "where", "why", "how", "can", "could",
    "will", "would", "should", "do", "does", "did",
    "not", "no", "more", "most", "very", "really",
    "also", "only", "all", "any", "some", "someone",
    "anyone", "everyone", "something", "anything",
    "please", "today", "now", "one", "like", "get",
    "got", "getting", "up", "down", "out", "just","telling","started","entire","everything"]
count_words={}
for msg in messages:
  words=msg["Messages"].lower().split()
  for word in words:
    if word in remove:
      continue
    if word not in count_words:
      count_words[word]=1
    else:
      count_words[word]=count_words[word]+1

top_words=[]
for word in count_words:
  top_words.append((word,count_words[word]))
top_words.sort(key=lambda x: x[1], reverse=True)
for i in range(6):
  print(top_words[i])

from datetime import datetime

print("*" * 100)
print("RESPONSE PATTERNS")

response_times = {}

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]
    if previous["Name"] == current["Name"]:
        continue
    previous_datetime = datetime.strptime(
        previous["Date"] + " " + previous["Time"],
        "%d/%m/%y %H:%M")
    current_datetime = datetime.strptime(
        current["Date"] + " " + current["Time"],
        "%d/%m/%y %H:%M")
    difference = current_datetime - previous_datetime
    if difference.total_seconds() > 24 * 60 * 60:
        continue

    seconds = difference.total_seconds()

    if current["Name"] not in response_times:
        response_times[current["Name"]] = []

    response_times[current["Name"]].append(seconds)
average_response = {}
for person in response_times:
    total = sum(response_times[person])
    number = len(response_times[person])
    average_response[person] = total / number

fastest_person = min(
    average_response,
    key=average_response.get)

slowest_person = max(
    average_response,
    key=average_response.get)
fastest_minutes = average_response[fastest_person] / 60
slowest_minutes = average_response[slowest_person] / 60
print("Fastest replier :", fastest_person,
      f"({fastest_minutes:.1f} minutes)")
print("Slowest replier :", slowest_person,
      f"({slowest_minutes:.1f} minutes)")

from datetime import datetime
print("*" * 100)
print("LONGEST SILENT STREAKS")

person_messages = {}
for msg in messages:
    person = msg["Name"]
    message_time = datetime.strptime(
        msg["Date"] + " " + msg["Time"],
        "%d/%m/%y %H:%M")
    if person not in person_messages:
        person_messages[person] = []

    person_messages[person].append(message_time)

longest_silence = {}
for person in person_messages:
    times = person_messages[person]
    times.sort()
    longest_gap = 0

    for i in range(1, len(times)):
        gap = times[i] - times[i - 1]
        if gap.total_seconds() > longest_gap:
            longest_gap = gap.total_seconds()

    longest_silence[person] = longest_gap

silent_person = max(
    longest_silence,
    key=longest_silence.get)
silent_seconds = longest_silence[silent_person]
silent_days = silent_seconds / (24 * 60 * 60)


print(f"{silent_person} was silent for {silent_days:.1f} days")
print("*"* 100)
print("PERSONALITY ARCHETYPES")
print(f"{fastest_person}-> THE LIVE AGENT (most active in group)")
print(f"{silent_person}-> THE AEROPLANE MODE (silent for {silent_days:.1f}days)")
night_activity={}
midnyt_hour=range(0,6)
for i in range(len(members)):
  total1=0
  for hr in midnyt_hour:
    total1=total1+heatmap[i][hour]
  night_activity[members[i]] = total1
night_owl = max(night_activity, key=night_activity.get)
print(f"{night_owl}-> THE NIGHT OWL (gets active at midnight)")

WHATSAPP CHAT REPORT :"Hostel bois 4ever"
Members: 6
Number of messages: 3174
Days: 60
****************************************************************************************************
MESSAGES PER PERSON
Rahul : 953
Priya : 718
Karan : 354
Neha : 635
Aman : 490
Vikas : 24
Busiest day: 04/05/24 (Highest messages: 76 )
****************************************************************************************************
Activity Heatmap
Hour:       00  01  02  03  04  05  06  07  08  09  10  11  12  13  14  15  16  17  18  19  20  21  22  23

Rahul     :  +   +   +   +   +   +   +   +   +   +   +   +   #   *   *   #   #   *   %   #   *   %   #   #  
Priya     :  .   .   .   .   .   .   +   *   #   %   %   %   %   #   #   *   *   #   #   %   #   *   *   +  
Karan     :  .   .   .   .   .   .   .   +   *   *   #   *   %   #   %   #   #   #   #   %   #   *   +   +  
Neha      :  .   .   .   .   .   *   +   +   #   %   %   *   #   #   *   +   #   %   %   %   #   *   *   *  
Aman      :  # 